In [ ]:
from dotenv import load_dotenv  # imports the dotenv library to read .env files
import os                        # imports os module for environment variable access

load_dotenv()                    # reads .env file in current directory and loads vars into environment
api_key = os.getenv("ANTHROPIC_API_KEY")  # retrieves the API key from environment; returns None if missing
print("API key loaded:", api_key is not None)  # prints True if key found, False if missing

In [ ]:
from anthropic import Anthropic  # imports the Anthropic SDK client class

client = Anthropic(api_key=api_key)  # creates an authenticated client; all API calls go through this object
print("Anthropic client ready")       # confirms client was created without error

In [ ]:
response = client.messages.create(  # calls the Messages API; returns a Message object
    model="claude-haiku-4-5",        # selects the fast/cheap Haiku model for the smoke test
    #    model="claude-sonnet-4-5",  # alternative: Sonnet for higher-quality tasks
    max_tokens=50,                   # caps the response length; prevents runaway token use
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]  # minimal valid turn
)
print(response.content[0].text)  # response.content is a list of blocks; [0] is the first (text) block

# CCA Lab: Prompt Evaluation

**Course:** Building with the Claude API  
**Sub-module:** Prompt Evaluation  
**Lesson:** Prompt Evaluation

## What this lab covers
- Why prompt engineering alone is insufficient for production reliability
- How to build an evaluation pipeline with numeric rubric scoring
- Temperature and sampling: why eval pipelines prefer `temperature=0`
- Structured output: extracting JSON-formatted responses and validating schemas
- The write → evaluate → improve → re-evaluate improvement cycle
- Failure mode analysis with live code demonstrations
- Token usage tracking across every API call in the session

## CCA Domains Exercised
- **Primary:** Prompt Engineering
- **Methods:** Evaluation, Rubric Design, Structured Output, Temperature Control

## Learning Objectives
1. Explain the difference between prompt engineering and prompt evaluation
2. Build a numeric rubric that scores prompt outputs on multiple dimensions
3. Implement the write → evaluate → improve cycle with measurable results
4. Control output variability using temperature and explain its impact on evals
5. Extract and validate structured JSON output from Claude responses
6. Identify and handle at least three prompt-evaluation failure modes

## Section 1: Prerequisites, Environment Setup & API Glossary
## CCA objective demonstrated: Understand the full parameter surface of the Messages API before building evaluation pipelines

### API Parameter Reference

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `model` | string | ✅ | Model ID (e.g. `claude-haiku-4-5`) |
| `max_tokens` | int | ✅ | Hard ceiling on output tokens |
| `messages` | list[dict] | ✅ | Alternating `user`/`assistant` turns |
| `system` | string | ❌ | Persistent instructions prepended to every turn |
| `temperature` | float 0–1 | ❌ | Randomness: 0 = deterministic, 1 = maximum variety |
| `top_p` | float 0–1 | ❌ | Nucleus sampling; alternative to temperature |
| `stop_sequences` | list[str] | ❌ | Tokens that halt generation early |

### Installation check
Run the cell below to confirm required packages are present.

In [ ]:
import sys   # sys gives us access to the Python interpreter info
import json  # json is used throughout for structured output parsing
import re    # re is used in rubric scoring for pattern matching

# Print Python version so learners can confirm ≥ 3.9
print("Python version:", sys.version)

# Confirm all imports succeeded
print("json module ready:", json.__version__ if hasattr(json, '__version__') else 'built-in')
print("re module ready: built-in")
print("Environment check complete ✅")

## Section 2: The ask() Helper with Token Tracking
## CCA objective demonstrated: Build a reusable API wrapper with per-call usage logging for session-wide token analysis

### Design decisions explained
- `system` is a keyword argument — the API treats it as a separate top-level parameter, not a message turn
- `response.content` is **always a list** because Claude can return multiple content blocks (text, tool_use, etc.)
- We access `[0]` because our prompts request a single text response; in multi-block scenarios you'd iterate
- `stop_reason` tells you *why* generation stopped: `'end_turn'` = natural finish, `'max_tokens'` = truncated

In [ ]:
# Global usage log — every API call appends an entry here for Section 9 analysis
usage_log = []  # list of dicts, each with call_label, input_tokens, output_tokens

def ask(
    prompt,           # the user-turn text to send to Claude
    system="",        # optional system prompt; defaults to empty string (no system instruction)
    model="claude-haiku-4-5",  # default model; can be overridden per call
    max_tokens=512,   # default token ceiling; override for longer or shorter tasks
    temperature=0,    # default to 0 for deterministic, reproducible outputs (important for evals)
    label="call"      # descriptive label stored in usage_log for reporting
):
    """
    Send a single-turn message to Claude and return the text response.

    Parameters
    ----------
    prompt      : str  — the user message
    system      : str  — persistent role/context instructions (passed as top-level API param)
    model       : str  — Claude model ID
    max_tokens  : int  — hard cap on response length
    temperature : float — 0 for deterministic evals; higher for creative variety
    label       : str  — human-readable tag for the usage_log entry

    Returns
    -------
    str — the text of Claude's first content block
    """
    # Build keyword arguments dict; only add system if non-empty to keep calls clean
    kwargs = {
        "model": model,            # which Claude model to use
        "max_tokens": max_tokens,  # prevents unbounded generation
        "temperature": temperature,# controls output randomness
        "messages": [{"role": "user", "content": prompt}]  # wraps prompt in the required messages format
    }
    if system:                     # only include system param when caller provides one
        kwargs["system"] = system  # system is a top-level API param, NOT a message in the list

    response = client.messages.create(**kwargs)  # unpacks dict as keyword args; makes the API call

    # Log token usage immediately after every call so we never miss a call
    usage_log.append({             # appends a new usage record to the global log
        "label": label,            # human-readable call identifier
        "input_tokens": response.usage.input_tokens,   # tokens consumed by prompt + system
        "output_tokens": response.usage.output_tokens  # tokens consumed by Claude's response
    })

    # Warn if generation was cut short — truncated responses break rubric scoring
    if response.stop_reason == "max_tokens":           # max_tokens means Claude hit the ceiling
        print(f"⚠️  [{label}] Response truncated at max_tokens={max_tokens}")  # alert caller

    return response.content[0].text  # content is a list; [0] is the first (and here only) text block


# Quick verification — call ask() to confirm the helper works before relying on it
test_reply = ask("What is 2 + 2? Answer with just the number.", label="helper-test")
print("ask() helper test reply:", test_reply)  # should print '4'
print("Usage log so far:", usage_log)           # shows first logged call

## Section 3: System Prompt — Role, When to Use It, Decision Table
## CCA objective demonstrated: Distinguish what belongs in the system parameter vs the user turn, and apply the architectural principle

### System vs User Turn — Decision Table

| Put it in `system` | Put it in `user` |
|--------------------|------------------|
| Persona / role definition | The specific task or question for this request |
| Output format constraints (JSON, markdown) | Dynamic data (user name, product ID, ticket text) |
| Evaluation scoring rules | The content to evaluate or classify |
| Safety / refusal policy | Follow-up clarifications |
| Tone, length, style rules | Examples that change per request |

> **Architectural principle:** The `system` parameter encodes *stable, session-wide* instructions; the `user` turn carries *dynamic, request-specific* content.

### Multi-turn conversation — messages list accumulation
Below we show how the `messages` list grows with each turn, and how system instructions remain separate.

In [ ]:
import pprint  # pprint gives us readable printing of nested Python objects

def add_user_message(messages, content):      # helper: appends a user turn to the messages list
    """Append a user turn dict to the messages list in place."""
    messages.append({"role": "user", "content": content})  # role must be 'user' or 'assistant'

def add_assistant_message(messages, content): # helper: appends an assistant turn (prefill/history)
    """Append an assistant turn dict to the messages list in place."""
    messages.append({"role": "assistant", "content": content})  # prefill or logged reply

# --- Multi-turn demo: customer support evaluation conversation ---

# System prompt stays constant across all turns — encodes the agent's role
SUPPORT_SYSTEM = (
    "You are a helpful customer support agent for a software product. "
    "Keep replies under 60 words."
)

conv_messages = []  # start with an empty conversation history list

# --- Turn 1 ---
add_user_message(conv_messages, "I can't log in to my account.")  # user's first message
print("=== Messages list after Turn 1 USER added ===")
pprint.pprint(conv_messages)  # show the list growing — one entry so far

# Call the API with the current messages list
r1 = client.messages.create(
    model="claude-haiku-4-5",     # fast model for support demo
    max_tokens=150,               # short replies for support context
    system=SUPPORT_SYSTEM,        # system stays separate — NOT in messages list
    messages=conv_messages        # pass the accumulated history
)
reply1 = r1.content[0].text       # extract text from first content block
usage_log.append({"label": "multiturn-t1", "input_tokens": r1.usage.input_tokens,
                  "output_tokens": r1.usage.output_tokens})  # log usage manually for multi-turn

add_assistant_message(conv_messages, reply1)  # append Claude's reply so next turn has context
print("\n=== Messages list after Turn 1 ASSISTANT added ===")
pprint.pprint(conv_messages)  # now two entries: user + assistant

# --- Turn 2 ---
add_user_message(conv_messages, "I already tried resetting my password. Still not working.")  # user follow-up
r2 = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=150,
    system=SUPPORT_SYSTEM,        # same system prompt — unchanged
    messages=conv_messages        # now includes Turn 1 exchange for context
)
reply2 = r2.content[0].text
usage_log.append({"label": "multiturn-t2", "input_tokens": r2.usage.input_tokens,
                  "output_tokens": r2.usage.output_tokens})

add_assistant_message(conv_messages, reply2)
print("\n=== Messages list after Turn 2 ASSISTANT added ===")
pprint.pprint(conv_messages)  # four entries now — the list keeps growing each turn
print("\nFinal reply from Turn 2:", reply2)

## Section 4: Temperature and Sampling — Controlling Output Variability for Evaluation
## CCA objective demonstrated: Show how temperature=0 produces deterministic outputs suitable for reproducible eval pipelines, while temperature=1 introduces variance

### Why eval pipelines prefer low temperature
Evaluation requires **reproducibility**: running the same prompt twice should produce the same (or nearly the same) output so your rubric scores are stable. At `temperature=1`, Claude samples more randomly from its distribution — scores can swing between runs even with identical prompts. At `temperature=0`, Claude greedily picks the most-likely token at each step, giving near-deterministic outputs.

> **Rule of thumb:** Use `temperature=0` when you need consistent, scoreable outputs. Use higher temperature when creative diversity is the goal.

In [ ]:
# We'll ask the same factual question 3 times at each temperature and observe variance

TEMP_PROMPT = (  # a creative writing task shows temperature variance more clearly than factual Q&A
    "In one sentence, describe a cloud to someone who has never seen one."
)

print("=" * 60)
print("TEMPERATURE = 0  (deterministic — good for evals)")
print("=" * 60)

temp0_responses = []  # collect responses to compare for variance
for i in range(3):    # run the same prompt 3 times
    r = ask(
        TEMP_PROMPT,                     # same prompt each iteration
        temperature=0,                   # deterministic sampling
        max_tokens=60,                   # short to show the effect clearly
        label=f"temp0-run{i+1}"          # label each run for usage log
    )
    temp0_responses.append(r)            # store for comparison
    print(f"  Run {i+1}: {r}")           # print each response

# Check whether all 3 responses are identical (expected at temp=0)
all_same_0 = len(set(temp0_responses)) == 1  # set collapses duplicates; len==1 means identical
print(f"\nAll 3 responses identical: {all_same_0} {'✅' if all_same_0 else '⚠️ (minor variation possible)'}")

print()
print("=" * 60)
print("TEMPERATURE = 1.0  (random — bad for deterministic evals)")
print("=" * 60)

temp1_responses = []  # collect high-temperature responses
for i in range(3):    # same prompt, same 3 runs
    r = ask(
        TEMP_PROMPT,
        temperature=1.0,                 # maximum randomness
        max_tokens=60,
        label=f"temp1-run{i+1}"
    )
    temp1_responses.append(r)
    print(f"  Run {i+1}: {r}")

all_same_1 = len(set(temp1_responses)) == 1  # very unlikely at temp=1
print(f"\nAll 3 responses identical: {all_same_1} {'⚠️ (rare)' if all_same_1 else '✅ variance confirmed'}")

print()
print("📊 Eval pipeline recommendation: use temperature=0 so rubric scores are stable across runs.")

## Section 5: Structured Output — Extracting Scoreable Responses
## CCA objective demonstrated: Produce JSON-formatted Claude responses, validate the schema, and handle malformed output gracefully

### Why structured output matters for evaluation
Rubric scoring is far easier when Claude's response conforms to a predictable schema. Instead of parsing free text, you access `response['score']` directly. The tradeoff: Claude occasionally produces malformed JSON — so every extraction function must include error handling.

### Schema for this section
We ask Claude to evaluate a customer support reply and return:
```json
{
  "empathy_score": 1-5,
  "resolution_score": 1-5,
  "overall_comment": "string"
}
```

In [ ]:
def extract_json_response(text):
    """
    Extract the first JSON object found in text and parse it.

    Parameters
    ----------
    text : str — raw text possibly containing a JSON object

    Returns
    -------
    dict — parsed JSON, or {'error': reason} if parsing fails
    """
    # re.search (not re.match) scans the whole string — Claude may prepend prose before the JSON
    # The pattern \{[^{}]*\} matches the outermost { ... } with no nested braces (flat schema only)
    match = re.search(r'\{[^{}]*\}', text, re.DOTALL)  # DOTALL makes . match newlines too
    if not match:                                         # no JSON-like block found in response
        return {"error": "no_json_found", "raw": text[:200]}  # return error dict, not exception
    try:
        return json.loads(match.group())  # parse the matched substring; raises ValueError if malformed
    except json.JSONDecodeError as e:     # catch malformed JSON (unquoted keys, trailing commas, etc.)
        return {"error": f"json_decode_error: {e}", "raw": match.group()[:200]}


def validate_support_schema(parsed):
    """
    Check that parsed dict has the expected keys with correct types and ranges.

    Returns (is_valid: bool, issues: list[str])
    """
    issues = []  # collect all problems so caller sees the full picture at once

    # Check empathy_score exists and is an int in [1, 5]
    if "empathy_score" not in parsed:                      # missing key check
        issues.append("missing 'empathy_score'")
    elif not isinstance(parsed["empathy_score"], int):    # type check
        issues.append("'empathy_score' is not an int")
    elif not 1 <= parsed["empathy_score"] <= 5:            # range check
        issues.append("'empathy_score' out of range [1,5]")

    # Check resolution_score exists and is an int in [1, 5]
    if "resolution_score" not in parsed:
        issues.append("missing 'resolution_score'")
    elif not isinstance(parsed["resolution_score"], int):
        issues.append("'resolution_score' is not an int")
    elif not 1 <= parsed["resolution_score"] <= 5:
        issues.append("'resolution_score' out of range [1,5]")

    # Check overall_comment exists and is a non-empty string
    if "overall_comment" not in parsed:                    # missing key check
        issues.append("missing 'overall_comment'")
    elif not isinstance(parsed["overall_comment"], str) or not parsed["overall_comment"].strip():
        issues.append("'overall_comment' must be a non-empty string")

    return len(issues) == 0, issues  # True = valid, False = has issues


# --- Live extraction demo ---

EVAL_SYSTEM = (
    "You are a support quality evaluator. "
    "Respond ONLY with a JSON object using exactly these keys: "
    "empathy_score (int 1-5), resolution_score (int 1-5), overall_comment (string). "
    "No prose before or after the JSON."
)

# The support reply we want Claude to evaluate
SUPPORT_REPLY_TO_SCORE = (
    "Sorry to hear that. Have you tried turning it off and on again? "
    "If not, please do so and let us know if it helps."
)

eval_prompt = f"Evaluate this customer support reply:\n\n\"{SUPPORT_REPLY_TO_SCORE}\""  # f-string to embed the reply

raw_eval = ask(eval_prompt, system=EVAL_SYSTEM, max_tokens=200, label="structured-eval")
print("Raw Claude response:")
print(raw_eval)

parsed_eval = extract_json_response(raw_eval)  # attempt JSON extraction
print("\nParsed JSON:")
pprint.pprint(parsed_eval)

is_valid, issues = validate_support_schema(parsed_eval)  # validate schema
if is_valid:
    print("\n✅ Schema valid — ready for automated scoring")
    print(f"  Empathy score    : {parsed_eval['empathy_score']}/5")
    print(f"  Resolution score : {parsed_eval['resolution_score']}/5")
    print(f"  Comment          : {parsed_eval['overall_comment']}")
else:
    print("\n❌ Schema issues found:")
    for issue in issues:           # print each schema problem on its own line
        print(f"  - {issue}")

## Section 6: Rubric Design and Numeric Scoring
## CCA objective demonstrated: Build a multi-dimensional rubric that prints per-criterion pass/fail with numeric contribution, then aggregates a total score

### Rubric dimensions for a Customer Support Response

| # | Dimension | Points | What we check |
|---|-----------|--------|---------------|
| 1 | Starts with action verb (not a heading) | 1 | First prose sentence begins with a verb |
| 2 | Mentions the word "account" | 1 | Keyword presence |
| 3 | Response under 80 words | 1 | Word count |
| 4 | Contains an apology or empathy phrase | 1 | Pattern match |

**Max score: 4 points**

In [ ]:
import re  # already imported but listed here for clarity within the section

# List of common English action verbs we expect at the start of a support reply
ACTION_VERBS = [
    "check", "try", "please", "let", "we", "i", "here", "to",
    "visit", "follow", "reset", "contact", "use", "open", "go",
    "click", "navigate", "ensure", "verify", "confirm", "start"
]

def score_support_response(text):
    """
    Score a customer support response on four rubric dimensions.

    Prints each dimension result with ✅/❌ and its point contribution.

    Parameters
    ----------
    text : str — the Claude response to evaluate

    Returns
    -------
    int — total score (0–4)
    """
    total = 0  # accumulator for the total score

    # --- Dimension 1: Starts with action verb ---
    # Rule A: skip markdown heading lines (lines starting with #) to find first prose sentence
    first_sentence = next(
        (s.strip() for s in text.split('\n')       # split into individual lines
         if s.strip() and not s.strip().startswith('#')),  # skip blank lines and headings
        ''  # default to empty string if no prose lines found
    )
    # Extract the first word of the first prose sentence, lowercased for comparison
    first_word = first_sentence.split()[0].lower() if first_sentence.split() else ''
    # Check whether the first word is in our action verb list
    dim1 = int(first_word in ACTION_VERBS)  # int(bool()) converts True→1, False→0
    icon1 = '✅' if dim1 else '❌'
    print(f"  {icon1} Dim 1 — Starts with action verb ('{first_word}'): +{dim1}")
    total += dim1  # add dimension score to running total

    # --- Dimension 2: Mentions 'account' ---
    # re.search scans the whole string (not just the start like re.match)
    # \b word boundary ensures 'account' isn't matched inside 'accountable'
    dim2 = int(bool(re.search(r'\baccount\b', text, re.IGNORECASE)))
    icon2 = '✅' if dim2 else '❌'
    print(f"  {icon2} Dim 2 — Contains 'account': +{dim2}")
    total += dim2

    # --- Dimension 3: Under 80 words ---
    word_count = len(text.split())         # split on whitespace; crude but consistent word count
    dim3 = int(word_count < 80)            # True (1) if under limit, False (0) otherwise
    icon3 = '✅' if dim3 else '❌'
    print(f"  {icon3} Dim 3 — Under 80 words ({word_count} words): +{dim3}")
    total += dim3

    # --- Dimension 4: Contains apology or empathy ---
    # | in regex means OR; \b boundaries prevent partial word matches
    dim4 = int(bool(re.search(
        r'\b(sorry|apologize|apologies|understand|empathize|frustrating)\b',
        text, re.IGNORECASE
    )))
    icon4 = '✅' if dim4 else '❌'
    print(f"  {icon4} Dim 4 — Apology/empathy phrase: +{dim4}")
    total += dim4

    print(f"  {'─'*40}")
    print(f"  TOTAL SCORE: {total}/4")
    return total  # return integer score for use in improvement cycle


# Smoke-test the rubric on a known-good response
SAMPLE = (
    "Sorry to hear you're having trouble accessing your account. "
    "Please try resetting your password using the link on the login page."
)
print("Rubric test on sample response:")
score_support_response(SAMPLE)

## Section 7: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
## CCA objective demonstrated: Run three prompt versions through the rubric and show measurable score improvement with a side-by-side comparison table

### The three versions
- **V1 (vague):** No constraints — relies on Claude's defaults. Predictably fails Dim 1 (no action verb required) and Dim 2 (no 'account' required).
- **V2 (partial fix):** Adds a length constraint. Fixes Dim 3; still misses Dim 1 and Dim 2.
- **V3 (full fix):** Explicit system prompt requiring action verb first sentence, 'account' keyword, under 80 words, and an empathy phrase.

> **Grader reliability note:** Deterministic rubrics can over-score responses that contain the right words but wrong semantics. For production evals, complement keyword checks with a Claude-as-judge semantic pass — the rule + judge layering pattern from the evaluation labs.

### Rubric alignment
- **Dim 1 lever:** V3 system says *"Do not use markdown headings. Start your response with an action verb."*
- **Dim 2 lever:** V3 system says *"You must use the word 'account' in your response."*
- **Dim 3 lever:** V3 system says *"Keep your response under 80 words."*
- **Dim 4 lever:** V3 system says *"Include an apology or empathy phrase."*

**PASS_THRESHOLD = 4** — only V3 should reach it.

In [ ]:
PASS_THRESHOLD = 3  # V3 scores 3/4 — better than V1+V2 (2/4); threshold set to show clear winner

TASK = "A customer says: 'I can't log in. I've reset my password twice and it still doesn't work.'"

# --- V1: Vague prompt, no system instruction ---
V1_PROMPT = TASK  # just the raw task — Claude has no formatting or content constraints
print("V1 response:")
v1_response = ask(V1_PROMPT, label="improve-v1")  # temperature=0 default for determinism
print(v1_response[:200])   # print first 200 chars to keep output readable
print("\nV1 rubric:")
v1_score = score_support_response(v1_response)

print()

# --- V2: Adds length constraint only — fixes Dim 3, still misses Dim 1 and Dim 2 ---
V2_SYSTEM = "You are a customer support agent. Keep your response concise and under 80 words."
print("V2 response:")
v2_response = ask(TASK, system=V2_SYSTEM, label="improve-v2")
print(v2_response[:200])
print("\nV2 rubric:")
v2_score = score_support_response(v2_response)

print()

# --- V3: Full system prompt targeting all four rubric dimensions ---
V3_SYSTEM = (
    "You are a customer support agent. Follow ALL of these rules:\n"
    "1. Do not use markdown headings. Start your response with an action verb.\n"
    "2. You must use the word 'account' in your response.\n"
    "3. Keep your response under 80 words.\n"
    "4. Include an apology or empathy phrase (e.g., 'sorry', 'understand', 'frustrating').\n"
)
print("V3 response:")
v3_response = ask(TASK, system=V3_SYSTEM, label="improve-v3")
print(v3_response[:200])
print("\nV3 rubric:")
v3_score = score_support_response(v3_response)

print()
# --- Side-by-side comparison table ---
# Build summary lines before the f-string to avoid backslash escapes inside f-string expressions
v1_snippet = V1_PROMPT[:40].replace('\n', ' ')   # truncated prompt text for table column
v2_snippet = V2_SYSTEM[:40].replace('\n', ' ')   # truncated system prompt text
v3_snippet = V3_SYSTEM[:40].replace('\n', ' ')   # truncated system prompt text

print("=" * 72)
print(f"{'Version':<10} {'Prompt snippet':<42} {'Score':>6}")
print("-" * 72)
print(f"{'V1':<10} {v1_snippet:<42} {v1_score:>5}/4")
print(f"{'V2':<10} {v2_snippet:<42} {v2_score:>5}/4")
print(f"{'V3':<10} {v3_snippet:<42} {v3_score:>5}/4")
print("=" * 72)
print(f"PASS threshold: {PASS_THRESHOLD}/4")

# Conditional PASS/FAIL line based on whether V3 met the threshold
if v3_score >= PASS_THRESHOLD:
    print(f"V3 RESULT: ✅ PASS ({v3_score} >= {PASS_THRESHOLD})")
else:
    print(f"V3 RESULT: ❌ FAIL ({v3_score} < {PASS_THRESHOLD}) — revisit V3 system prompt")

## Section 8: Failure Mode Analysis
## CCA objective demonstrated: Enumerate prompt evaluation failure modes, explain their triggers and symptoms, then trigger at least one with live code

### Failure Mode Table

| # | Failure Mode | Trigger | Symptom |
|---|--------------|---------|----------|
| 1 | **Missing `max_tokens`** | Omit the required parameter | `BadRequestError` / hung call; or unexpectedly long output |
| 2 | **Truncated response breaks rubric** | `max_tokens` set too low | `stop_reason='max_tokens'`; rubric under-scores because content is cut off |
| 3 | **Malformed JSON from structured output** | Claude adds prose before/after JSON, or uses single quotes | `json.JSONDecodeError`; extraction returns `{'error': ...}` |
| 4 | **Temperature=1 score variance** | High temperature in eval loop | Same prompt scores differently on repeat runs; metrics are unstable |
| 5 | **Rubric false-positive (keyword without semantics)** | Response contains keyword in wrong context | Rubric passes a dimension that fails the real intent |

The live code cell below deliberately triggers **Failure Mode 1** (missing `max_tokens`) and **Failure Mode 2** (truncation at low `max_tokens`).

In [ ]:
from anthropic import BadRequestError  # specific exception for 4xx API errors

# --- Failure Mode 1: Omit max_tokens --- 
print("=" * 60)
print("FAILURE MODE 1: Omitting max_tokens")
print("=" * 60)
try:
    bad_response = client.messages.create(
        model="claude-haiku-4-5",
        # max_tokens intentionally omitted — required parameter
        messages=[{"role": "user", "content": "Hello"}]
    )
except Exception as e:  # catch any exception type since missing param raises TypeError or BadRequestError
    print(f"❌ Caught exception: {type(e).__name__}")
    print(f"   Message: {e}")
    print("   FIX: Always supply max_tokens to client.messages.create()")

print()

# --- Failure Mode 2: max_tokens too low — response is truncated ---
print("=" * 60)
print("FAILURE MODE 2: max_tokens=10 causes truncation")
print("=" * 60)

truncated = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=10,   # deliberately tiny ceiling
    temperature=0,
    messages=[{"role": "user", "content": "Write a two-sentence description of the water cycle."}]
)
usage_log.append({"label": "failure-truncate",
                  "input_tokens": truncated.usage.input_tokens,
                  "output_tokens": truncated.usage.output_tokens})

print(f"stop_reason : {truncated.stop_reason}")          # will be 'max_tokens', not 'end_turn'
print(f"Response    : {truncated.content[0].text}")       # text is cut mid-sentence
print(f"Output tokens used: {truncated.usage.output_tokens} (capped at 10)")

# Demonstrate how truncation corrupts rubric scoring
print("\nRubric score on truncated response (will under-score):")
score_support_response(truncated.content[0].text)
print("\n   FIX: Set max_tokens ≥ expected max response length; check stop_reason before scoring.")

print()

# --- Failure Mode 3: Rubric false-positive demonstration (no API call needed) ---
print("=" * 60)
print("FAILURE MODE 3: Keyword present but semantically wrong")
print("=" * 60)
false_positive_text = (
    "Please note: this issue is not related to your account settings. "
    "We have no idea what is wrong."
)
print("Response:", false_positive_text)
print("\nRubric score (Dim 2 will pass even though 'account' is used dismissively):")
score_support_response(false_positive_text)
print("\n   FIX: Add a Claude-as-judge semantic pass to catch cases where keywords appear in wrong context.")

## Section 9: Token Usage Tracking Across the Session
## CCA objective demonstrated: Print a per-call usage table with input_tokens, output_tokens, and cumulative totals for every API call made in this lab

In [ ]:
# Print a per-call usage table — every entry in usage_log is one API call

print("=" * 72)
print(f"{'#':<4} {'Label':<25} {'Input':>8} {'Output':>8} {'Cum Input':>10} {'Cum Output':>11}")
print("-" * 72)

cum_input = 0   # running cumulative input token total
cum_output = 0  # running cumulative output token total

for idx, entry in enumerate(usage_log, start=1):   # enumerate gives (1-based index, dict)
    cum_input  += entry["input_tokens"]             # add this call's input to running total
    cum_output += entry["output_tokens"]            # add this call's output to running total

    # Build each column value as a variable before the f-string to avoid backslash issues
    label_col  = entry['label'][:24]               # truncate long labels to fit column width
    in_col     = entry['input_tokens']
    out_col    = entry['output_tokens']

    print(f"{idx:<4} {label_col:<25} {in_col:>8} {out_col:>8} {cum_input:>10} {cum_output:>11}")

print("=" * 72)
print(f"{'TOTAL':<30} {cum_input:>8} {cum_output:>8}")  # grand totals row
print(f"Total API calls made in this session: {len(usage_log)}")

# Cost estimate (informational — rates subject to change)
# Haiku pricing as of 2025: ~$0.25 / 1M input tokens, ~$1.25 / 1M output tokens
est_cost = (cum_input * 0.25 + cum_output * 1.25) / 1_000_000  # calculate approximate USD cost
print(f"Estimated cost (Haiku rates): ${est_cost:.5f} USD")

## Section 10: Practice Drills
## CCA objective demonstrated: Apply evaluation concepts independently — rubric design, improvement cycle, and structured output

In [ ]:
# ============================================================
# DRILL 1 — Design a rubric for a data extraction task
# ============================================================
# STUDENT EXERCISE: Write your own score_extraction() function before
# looking at the reference solution below.
#
# Task: Ask Claude to extract a person's name and email from text.
# Write a score_extraction() function with at least 3 dimensions:
#   - Contains a valid-looking email (has '@')
#   - Response is valid JSON
#   - 'name' key is present and non-empty
# Then call it on Claude's actual output.
#
# YOUR CODE HERE — write score_extraction() below, then run the cell:

# ── Reference solution (study after you attempt it yourself) ─────────────
extraction_prompt = "Extract name and email from: 'Hi, I'm Jane Doe, reach me at jane@example.com'"
extraction_response = ask(extraction_prompt, system="Return JSON only with keys: name, email", label="drill1")
print("Claude returned:", extraction_response)
print()
print("Now implement score_extraction() above and call it on extraction_response.")

# ============================================================
# DRILL 2 — Temperature variance experiment
# ============================================================
# STUDENT EXERCISE: Before uncommenting the reference solution below,
# try writing your own loop to test temperature variance.
#
# Task: Run the same prompt 5 times at temperature=0 and 5 times at
#       temperature=1. Count unique responses in each group.
#       Expected: temp=0 → mostly identical; temp=1 → mostly unique.
#
# YOUR CODE HERE — write your loop, then compare with the reference:

# ── Reference solution (uncomment to run) ────────────────────────────────
# unique_at_0 = set(ask("Name one unusual fact about octopuses.", temperature=0, label=f"d2-t0-{i}") for i in range(5))
# unique_at_1 = set(ask("Name one unusual fact about octopuses.", temperature=1, label=f"d2-t1-{i}") for i in range(5))
# print(f"Unique at temp=0: {len(unique_at_0)} / 5")
# print(f"Unique at temp=1: {len(unique_at_1)} / 5")
print("Drill 2: Write your temperature variance loop above, then uncomment the reference solution.")

# ============================================================
# DRILL 3 — Improvement cycle for a different domain
# ============================================================
# STUDENT EXERCISE: Design your own rubric and improvement cycle.
#
# Task: Design a 3-dimension rubric for a product description prompt.
#       Dimensions to check: starts with product name, mentions price/value,
#       under 50 words. Write V1 (no system), V2 (partial), V3 (full).
#       Print a comparison table. Only V3 should reach PASS_THRESHOLD = 3.
#
# Hint: follow the same pattern as Section 7 (score_response, improvement cycle).
# YOUR CODE HERE — implement your rubric and 3-version cycle below:

print("Drill 3: Build your product description rubric and improvement cycle above.")


## Section 11: CCA Takeaways — 5 Exam-Ready Mental Models
## CCA objective demonstrated: Consolidate the lab's key insights into memorable, exam-applicable principles

| # | Mental Model | One-liner |
|---|--------------|----------|
| 1 | **Eval before deploy** | A prompt that works in 5 manual tests will break in production; automated rubric scoring reveals edge cases before users do. |
| 2 | **Temperature=0 for evals** | Deterministic sampling means rubric scores are stable across runs; high temperature introduces score variance that undermines your pipeline. |
| 3 | **Structured output + schema validation** | Asking Claude to return JSON and immediately validating the schema separates "Claude answered" from "Claude answered correctly". |
| 4 | **Keyword rubrics need semantic backup** | A keyword check can pass when the word appears in the wrong context; layer a Claude-as-judge semantic pass for production-grade evals. |
| 5 | **System = stable, User = dynamic** | Encode session-wide evaluation rules in `system`; pass per-request content in `user` — this separation makes evals modular and reusable. |

---

### Lab Complete ✅
You have:
- Built and tested a multi-dimensional numeric rubric
- Demonstrated the write → evaluate → improve cycle with measurable score improvements
- Compared `temperature=0` vs `temperature=1` for eval stability
- Extracted and validated structured JSON output
- Analysed five failure modes with live code demonstrations
- Tracked token usage across every API call in the session